# Wildfire Intensity Classification using Machine Learning

This notebook presents a portfolio-friendly machine learning workflow for predicting wildfire intensity levels using environmental, weather, satellite-derived, and geographic features.

The original coursework dataset is **not included** in this repository because it was provided for assessment use only. To run the notebook, place the permitted dataset files inside a local `data/` folder.

## Project task

Predict the wildfire fire intensity class for each fire event:

- Low
- Moderate
- High
- Extreme

This is a supervised multi-class classification problem. The workflow compares three models:

- Decision Tree
- Support Vector Machine
- Neural Network

The main evaluation metrics are **accuracy** and **macro F1-score**. Macro F1-score is especially important because the fire intensity classes are imbalanced.


## 1. Setup

The notebook uses common Python machine learning libraries. Outputs are intentionally not saved in this public version, and no restricted data files are included.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load data

Place the following files in a local `data/` folder before running this notebook:

- `wildfire_cls_train_full.csv`
- `wildfire_cls_test_features.csv`

The public GitHub repository does not include these files.


In [ ]:
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "wildfire_cls_train_full.csv"
TEST_PATH = DATA_DIR / "wildfire_cls_test_features.csv"

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        "Dataset files were not found. Place the permitted CSV files in a local 'data/' folder. "
        "The dataset is not included in this repository due to coursework data-use restrictions."
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)
print("Columns loaded successfully.")


## 3. Task and dataset overview

The target variable is `fire_intensity`. The input features include geographic coordinates, acquisition time, season, region, country, fire type, satellite/sensor information, brightness temperature, confidence level, maximum temperature, wind speed, precipitation, and humidity.

The dataset contains both numerical and categorical variables, so preprocessing is required before modelling.


In [ ]:
TARGET = "fire_intensity"

if TARGET not in train_df.columns:
    raise ValueError(f"Expected target column '{TARGET}' was not found in the training data.")

print("Training columns:")
print(train_df.columns.tolist())

print("\nData types:")
print(train_df.dtypes)

print("\nMissing values in training data:")
print(train_df.isna().sum())

print("\nTarget class distribution:")
print(train_df[TARGET].value_counts())


## 4. Exploratory data analysis

The EDA focuses on class imbalance, missing values, and key feature relationships. Raw data rows are not displayed in this public notebook.


In [ ]:
# Class distribution
class_counts = train_df[TARGET].value_counts()

plt.figure(figsize=(7, 4))
class_counts.plot(kind="bar")
plt.title("Distribution of Fire Intensity Classes")
plt.xlabel("Fire Intensity")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Missing value summary
missing = train_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(7, 4))
missing.plot(kind="bar")
plt.title("Missing Values by Feature")
plt.xlabel("Feature")
plt.ylabel("Missing Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Relationship between brightness temperature and fire intensity
if "brightness_k" in train_df.columns:
    plt.figure(figsize=(8, 4))
    train_df.boxplot(column="brightness_k", by=TARGET)
    plt.title("Brightness Temperature by Fire Intensity")
    plt.suptitle("")
    plt.xlabel("Fire Intensity")
    plt.ylabel("Brightness Temperature (K)")
    plt.tight_layout()
    plt.show()


### EDA interpretation

The original analysis showed that the target classes were imbalanced, with Moderate and High events appearing more frequently than Low and Extreme events. This means accuracy alone may hide weak performance on minority classes, so macro F1-score is used alongside accuracy.

Brightness temperature was one of the clearer indicators of fire intensity, which is reasonable because stronger fires tend to produce stronger thermal signals. Other environmental variables, such as temperature, wind, precipitation, and humidity, can contribute useful information but do not individually separate all classes.


## 5. Feature preparation

The `acq_date` column is converted to a date format so that missing `month` values can be recovered from it. The raw date column is then removed before modelling to avoid high-cardinality date encoding.


In [ ]:
def prepare_features(df: pd.DataFrame, target: str | None = None):
    # Prepare wildfire features for modelling without displaying restricted raw data.
    df = df.copy()

    y = None
    if target is not None and target in df.columns:
        y = df[target].copy()
        df = df.drop(columns=[target])

    if "acq_date" in df.columns:
        acq_date = pd.to_datetime(df["acq_date"], errors="coerce")
        if "month" in df.columns:
            df["month"] = df["month"].fillna(acq_date.dt.month)
        df = df.drop(columns=["acq_date"])

    return df, y

X, y = prepare_features(train_df, target=TARGET)
X_test, _ = prepare_features(test_df)

print("Prepared training feature shape:", X.shape)
print("Prepared test feature shape:", X_test.shape)


In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## 6. Evaluation framework

A stratified 80/20 train-validation split is used so that the validation set keeps a similar class distribution to the training data. This supports fairer model comparison in an imbalanced multi-class task.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training split:", X_train.shape)
print("Validation split:", X_val.shape)
print("Training class distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("\nValidation class distribution (%):")
print((y_val.value_counts(normalize=True) * 100).round(2))


## 7. Preprocessing pipeline

Numerical features are median-imputed and standardised. Categorical features are imputed using the most frequent value and one-hot encoded. This pipeline is reused consistently for all models.


In [ ]:
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot_encoder),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val):
    # Return train and validation accuracy and macro F1-score.
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    return {
        "train_accuracy": accuracy_score(y_train, train_pred),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "train_f1_macro": f1_score(y_train, train_pred, average="macro"),
        "val_f1_macro": f1_score(y_val, val_pred, average="macro"),
    }

model_results = []


## 8. Decision Tree model

The Decision Tree is interpretable but can overfit easily when the tree is too deep. The main hyperparameter investigated here is `max_depth`.


In [ ]:
depth_values = [2, 3, 4, 5, 6, 8, 10, None]
dt_depth_results = []

for depth in depth_values:
    dt_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)),
    ])
    dt_model.fit(X_train, y_train)
    result = evaluate_model(dt_model, X_train, y_train, X_val, y_val)
    result["max_depth"] = depth
    dt_depth_results.append(result)

dt_depth_results_df = pd.DataFrame(dt_depth_results)
dt_depth_results_df


In [ ]:
# Plot Decision Tree validation performance across max_depth values
plot_df = dt_depth_results_df.copy()
plot_df["max_depth"] = plot_df["max_depth"].astype(str)

plt.figure(figsize=(8, 4))
plt.plot(plot_df["max_depth"], plot_df["train_accuracy"], marker="o", label="Training Accuracy")
plt.plot(plot_df["max_depth"], plot_df["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("Decision Tree Accuracy vs max_depth")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(plot_df["max_depth"], plot_df["train_f1_macro"], marker="o", label="Training Macro F1")
plt.plot(plot_df["max_depth"], plot_df["val_f1_macro"], marker="o", label="Validation Macro F1")
plt.title("Decision Tree Macro F1 vs max_depth")
plt.xlabel("max_depth")
plt.ylabel("Macro F1-score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
dt_best = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
])

dt_best.fit(X_train, y_train)
dt_best_results = evaluate_model(dt_best, X_train, y_train, X_val, y_val)

dt_best_results


In [ ]:
dt_val_pred = dt_best.predict(X_val)
cm = confusion_matrix(y_val, dt_val_pred, labels=dt_best.classes_)

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dt_best.classes_).plot(xticks_rotation=45)
plt.title("Decision Tree Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_val, dt_val_pred))


In [ ]:
model_results.append({
    "model": "Decision Tree",
    "best_setting": "max_depth=5",
    **dt_best_results,
})


### Decision Tree interpretation

In the original experiment, deeper Decision Trees fitted the training data very strongly but did not generalise well. Limiting the tree depth reduced overfitting, but the final validation macro F1-score remained moderate. This suggests that a single Decision Tree is interpretable but may be too simple or unstable for this problem.


## 9. Support Vector Machine model

The SVM uses an RBF kernel to learn non-linear decision boundaries. The main hyperparameters investigated are `C` and `gamma`.


In [ ]:
svm_candidates = [
    {"C": 0.5, "gamma": "scale"},
    {"C": 1, "gamma": "scale"},
    {"C": 2, "gamma": "scale"},
    {"C": 5, "gamma": "scale"},
    {"C": 2, "gamma": 0.01},
    {"C": 2, "gamma": 0.1},
]

svm_results = []

for params in svm_candidates:
    svm_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC(kernel="rbf", C=params["C"], gamma=params["gamma"], random_state=RANDOM_STATE)),
    ])
    svm_model.fit(X_train, y_train)
    result = evaluate_model(svm_model, X_train, y_train, X_val, y_val)
    result.update(params)
    svm_results.append(result)

svm_results_df = pd.DataFrame(svm_results)
svm_results_df


In [ ]:
plt.figure(figsize=(8, 4))
labels = [f"C={row.C}, gamma={row.gamma}" for row in svm_results_df.itertuples()]
plt.plot(labels, svm_results_df["val_accuracy"], marker="o", label="Validation Accuracy")
plt.plot(labels, svm_results_df["val_f1_macro"], marker="o", label="Validation Macro F1")
plt.title("SVM Validation Performance across Hyperparameters")
plt.xlabel("Hyperparameter setting")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
svm_best = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(kernel="rbf", C=2, gamma="scale", random_state=RANDOM_STATE)),
])

svm_best.fit(X_train, y_train)
svm_best_results = evaluate_model(svm_best, X_train, y_train, X_val, y_val)

svm_best_results


In [ ]:
svm_val_pred = svm_best.predict(X_val)
cm = confusion_matrix(y_val, svm_val_pred, labels=svm_best.classes_)

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=svm_best.classes_).plot(xticks_rotation=45)
plt.title("SVM Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_val, svm_val_pred))


In [ ]:
model_results.append({
    "model": "SVM",
    "best_setting": "RBF, C=2, gamma=scale",
    **svm_best_results,
})


### SVM interpretation

In the original experiment, the SVM achieved the strongest validation accuracy. However, its macro F1-score remained lower than the Neural Network, suggesting that it still struggled with some minority classes in the imbalanced target distribution.


## 10. Neural Network model

The Neural Network can model more flexible non-linear relationships. The experiments investigate learning rate and hidden layer size. A loss curve is also plotted to inspect convergence behaviour.


In [ ]:
learning_rates = [0.0001, 0.0005, 0.001, 0.005]
nn_lr_results = []

for lr in learning_rates:
    nn_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=(50,),
            learning_rate_init=lr,
            max_iter=200,
            random_state=RANDOM_STATE,
        )),
    ])
    nn_model.fit(X_train, y_train)
    result = evaluate_model(nn_model, X_train, y_train, X_val, y_val)
    result["learning_rate"] = lr
    nn_lr_results.append(result)

nn_lr_results_df = pd.DataFrame(nn_lr_results)
nn_lr_results_df


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(nn_lr_results_df["learning_rate"].astype(str), nn_lr_results_df["val_accuracy"], marker="o", label="Validation Accuracy")
plt.plot(nn_lr_results_df["learning_rate"].astype(str), nn_lr_results_df["val_f1_macro"], marker="o", label="Validation Macro F1")
plt.title("Neural Network Validation Performance vs Learning Rate")
plt.xlabel("Learning Rate")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
hidden_layer_values = [(25,), (50,), (100,), (50, 25), (100, 50)]
nn_layer_results = []

for layers in hidden_layer_values:
    nn_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=layers,
            learning_rate_init=0.0005,
            max_iter=200,
            random_state=RANDOM_STATE,
        )),
    ])
    nn_model.fit(X_train, y_train)
    result = evaluate_model(nn_model, X_train, y_train, X_val, y_val)
    result["hidden_layers"] = str(layers)
    nn_layer_results.append(result)

nn_layer_results_df = pd.DataFrame(nn_layer_results)
nn_layer_results_df


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(nn_layer_results_df["hidden_layers"], nn_layer_results_df["train_f1_macro"], marker="o", label="Training Macro F1")
plt.plot(nn_layer_results_df["hidden_layers"], nn_layer_results_df["val_f1_macro"], marker="o", label="Validation Macro F1")
plt.title("Neural Network Macro F1 vs Hidden Layer Size")
plt.xlabel("Hidden Layer Size")
plt.ylabel("Macro F1-score")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
nn_best = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(50,),
        learning_rate_init=0.0005,
        max_iter=200,
        random_state=RANDOM_STATE,
    )),
])

nn_best.fit(X_train, y_train)
nn_best_results = evaluate_model(nn_best, X_train, y_train, X_val, y_val)

nn_best_results


In [ ]:
loss_curve = nn_best.named_steps["classifier"].loss_curve_

plt.figure(figsize=(8, 4))
plt.plot(loss_curve)
plt.title("Neural Network Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
nn_val_pred = nn_best.predict(X_val)
cm = confusion_matrix(y_val, nn_val_pred, labels=nn_best.classes_)

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=nn_best.classes_).plot(xticks_rotation=45)
plt.title("Neural Network Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_val, nn_val_pred))


In [ ]:
model_results.append({
    "model": "Neural Network",
    "best_setting": "hidden_layer=(50,), lr=0.0005",
    **nn_best_results,
})


### Neural Network interpretation

In the original experiment, the Neural Network achieved the best validation macro F1-score. This made it the preferred final model because macro F1-score is more appropriate for an imbalanced classification task where minority classes such as Low and Extreme should not be ignored.


## 11. Model comparison and final selection

The tuned models are compared using the same validation set and the same metrics.


In [ ]:
model_results_df = pd.DataFrame(model_results)
model_results_df


In [ ]:
comparison_plot = model_results_df.set_index("model")[["val_accuracy", "val_f1_macro"]]

comparison_plot.plot(kind="bar", figsize=(8, 4))
plt.title("Validation Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(["Validation Accuracy", "Validation Macro F1"])
plt.grid(axis="y")
plt.tight_layout()
plt.show()


### Portfolio result summary

The original coursework experiment produced the following validation results:

| Model | Best setting | Validation Accuracy | Validation Macro F1 |
|---|---:|---:|---:|
| Decision Tree | `max_depth=5` | 0.4666 | 0.3462 |
| SVM | `RBF, C=2, gamma=scale` | 0.4793 | 0.3465 |
| Neural Network | `hidden_layer=(50,), lr=0.0005` | 0.4654 | 0.4075 |

The SVM had the highest validation accuracy, but the Neural Network achieved the highest validation macro F1-score. Because the dataset was imbalanced and minority classes were important, the Neural Network was selected as the final model.


## 12. Responsible use and limitations

This model should be treated as a decision-support tool, not a replacement for expert wildfire assessment. In real-world use, model outputs should be reviewed alongside satellite imagery, weather forecasts, local emergency reports, domain expertise, and operational constraints.

Key limitations include:

- class imbalance, especially for minority intensity classes
- possible uncertainty in satellite-derived features
- limited generalisation beyond the represented regions and countries
- moderate validation performance, meaning predictions should not be used alone for high-stakes decisions


## 13. Optional final model training

This section trains the selected Neural Network on the full available training dataset. It does not save predictions in this public version.


In [ ]:
final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(50,),
        learning_rate_init=0.0005,
        max_iter=200,
        random_state=RANDOM_STATE,
    )),
])

final_model.fit(X, y)
print("Final Neural Network model trained successfully.")


## 14. Conclusion

This project compared Decision Tree, SVM, and Neural Network models for wildfire intensity classification. The workflow included missing value handling, feature preprocessing, stratified validation, hyperparameter tuning, performance curves, confusion matrices, and responsible AI discussion.

The Neural Network was selected as the preferred model because it provided the best macro F1-score, giving a more balanced evaluation across all four fire intensity classes.
